In [1]:
import os
os.chdir('../')  # coconut root dir

In [2]:
from coconut.configs import TRM
from coconut.load_model import resume_model, load_new_model, tie_embeddings
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_15/pytorch_model.safetensors'
# load_model_path='./outputs/trm-qwen3-0.6b_20251021-131708/checkpoint_8/pytorch_model.safetensors'
load_model_path='./outputs/trm-qwen3-0.6b_20251022-065910/checkpoint_24/pytorch_model.safetensors'

# could load from coconut_config.toml
conf = TRM(
    # load_model_path=load_model_path
    # model_id = "suayptalha/Qwen3-0.6B-Math-Expert"
    model_id="Qwen/Qwen3-0.6B"
    )

# conf.model_id = '../'+conf.model_id
conf

/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/media/wassname/SGIronWolf/projects5/2025/fbai_coconut/.venv/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` m

TRM(project='coconut', save_path='outputs/', name='trm-qwen3-0.6b', model_id='Qwen/Qwen3-0.6B', only_eval=False, load_model_path='', resume_epochs=8, replacement_method='supressed[0.75:]', use_position_ids=True, bf16=True, bf16_weight=False, opt_8b=False, cot_epochs=0, epochs_per_stage=8, max_latent_stage=3, num_epochs=25, batch_size_training=12, gradient_accumulation_steps=64, lr=0.0001, weight_decay=0.1, grad_clip=1.0, scheduler='cosine', debug=False, seed=42, reset_optimizer=False, loss_seq_vcr=False, n_detached_recursions=2, use_trm=True, load_in_4bit=True, load_in_8bit=False, max_size=20000, c_thought=1, pad_latent_to_max=True, uniform_prob=0.0, train_path='data/gsm_train.json', val_path='data/gsm_valid.json', system_prompt='', latent_token_id=None, bot_token_id=None, eot_token_id=None, eval_first_epoch=False, trm_h_cycles=3, trm_l_cycles=6, trm_l_layers=2, trm_hidden_size=None, trm_num_heads=8, trm_expansion=2.67, trm_transcoder_layers=1, trm_svd_init=False, trm_persistent_steeri

In [3]:
import safetensors
# model, tokenizer = load_new_model(conf, device, dtype)
model, tokenizer = load_new_model(conf, device, dtype)
tie_embeddings(model, tokenizer)
# model, tokenizer = resume_model(conf, device, dtype)
state_dict = safetensors.torch.load_file(load_model_path)
print('loading', state_dict.keys())
model.load_state_dict(state_dict, strict=False);

2025-10-22 14:32:18.995 | INFO     | coconut.load_model:load_new_model:49 - TRM mode: loading model with quantization
2025-10-22 14:32:18.996 | INFO     | coconut.load_model:load_new_model:53 - Loading in 4bit
`torch_dtype` is deprecated! Use `dtype` instead!
2025-10-22 14:32:22.214 | INFO     | coconut.coconut:__init__:83 - Initializing TRM adapter for frozen LLM
2025-10-22 14:32:22.349 | INFO     | coconut.coconut:__init__:97 - TRM adapter dtype: torch.bfloat16
2025-10-22 14:32:22.350 | INFO     | coconut.coconut:__init__:99 - Freezing base LLM parameters


loading dict_keys(['model.lm_head.weight', 'trm.l_net.layers.0.mlp.down_proj.weight', 'trm.l_net.layers.0.mlp.gate_up_proj.weight', 'trm.l_net.layers.0.self_attn.o_proj.weight', 'trm.l_net.layers.0.self_attn.qkv_proj.weight', 'trm.l_net.layers.1.mlp.down_proj.weight', 'trm.l_net.layers.1.mlp.gate_up_proj.weight', 'trm.l_net.layers.1.self_attn.o_proj.weight', 'trm.l_net.layers.1.self_attn.qkv_proj.weight', 'trm.q_head.bias', 'trm.q_head.weight', 'trm.transcoder.final_proj.weight', 'trm.transcoder.mlp.0.down_proj.weight', 'trm.transcoder.mlp.0.gate_up_proj.weight', 'trm.zH_init', 'trm.zL_init'])


In [4]:
# ds
from coconut.dataset import (
    CoconutCollator,
    get_cot_latent_dataset,
    get_dataset,
    get_question_only_latent_dataset,
)
max_size = 32

base_dataset_valid = get_dataset(
    conf.val_path,
    tokenizer,
    max_size=max_size // 30 + 3,
    drop_unused=False,
    system_prompt=conf.system_prompt,
)

from scripts.run import run_ratio_eval
from coconut.eval import evaluate, get_answer_perplexity, get_answer_preference

stage=1
model.to(device=device, dtype=dtype)
run_ratio_eval(
    model,
    tokenizer,
    base_dataset_valid,
    conf,
    stage,
)


num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


tokenize_sample: data/gsm_valid.json (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

2025-10-22 14:32:28.731 | DEBUG    | coconut.dataset:get_dataset:86 - Example row:
2025-10-22 14:32:28.732 | DEBUG    | coconut.dataset:get_dataset:87 - question_tokenized
2025-10-22 14:32:28.732 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[0] <<4-2=2>>
2025-10-22 14:32:28.732 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[1] <<2/.5=4>>
2025-10-22 14:32:28.732 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[2] <<12/4=3>>
2025-10-22 14:32:28.733 | DEBUG    | coconut.dataset:get_dataset:89 - steps_tokenized[3] <<100*3=300>>
2025-10-22 14:32:28.733 | DEBUG    | coconut.dataset:get_dataset:90 - answer_tokenized ### 300
<|im_end|>

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


process_dataset: cot_latent_1 (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

2025-10-22 14:32:30.227 | INFO     | coconut.eval:get_answer_preference:394 - ratio nll_ans/nll_corrupted_ans = 0.8738


{'eval/ratios': np.float64(0.8737762741123904),
 'eval/nll_chos_avg': np.float64(1.3194823197891101),
 'eval/nll_refs_avg': np.float64(1.45441323516044)}

In [5]:
latent_id = tokenizer.convert_tokens_to_ids("<|latent|>")
bot_id = tokenizer.convert_tokens_to_ids("<|start-latent|>")
eot_id = tokenizer.convert_tokens_to_ids("<|end-latent|>")
collator = CoconutCollator(tokenizer, latent_id=latent_id, label_pad_token_id=-100)
max_new_tokens = 64

dataset_gen_val = get_question_only_latent_dataset(
    stage,
    base_dataset_valid,
    conf,
    bot_id,
    latent_id,
    eot_id,
    # drop_unused=False,
)
valid_gen_dataloader = torch.utils.data.DataLoader(
    dataset_gen_val,
    num_workers=6,
    pin_memory=True,
    batch_size=conf.batch_size_training,
    collate_fn=collator,
)
r = evaluate(
    valid_gen_dataloader,
    model,
    tokenizer,
    base_dataset_valid,
    max_new_tokens=max_new_tokens,
    name=f"eval_{load_model_path}",
    dtype=dtype,
    device=device,
)

num_proc must be <= 4. Reducing num_proc to 4 for dataset of size 4.


process_dataset: q_latent_1 (num_proc=4):   0%|          | 0/4 [00:00<?, ? examples/s]

Test Accuracy eval_./outputs/trm-qwen3-0.6b_20251022-065910/checkpoint_24/pytorch_model.safetensors:   0%|    …

2025-10-22 14:32:30.672 | INFO     | coconut.eval:evaluate:40 - Starting evaluation eval_./outputs/trm-qwen3-0.6b_20251022-065910/checkpoint_24/pytorch_model.safetensors
2025-10-22 14:32:34.795 | DEBUG    | coconut.eval:evaluate:98 - Q #0: Question: `John cuts his grass to 2 inches.  It grows .5 inches per month.  When it gets to 4 inches he cuts it back down to 2 inches.  It cost $100 to get his grass cut.  How much does he pay per year?`.
Full question: `<|im_start|>system
	
	<|im_start|>user
	John cuts his grass to 2 inches.  It grows .5 inches per month.  When it gets to 4 inches he cuts it back down to 2 inches.  It cost $100 to get his grass cut.  How much does he pay per year?
	<|im_start|>assistant
	<think>
	
	</think>
	
	<|start-latent|><|latent|><|end-latent|>`.
Full llm output: `['-->\n\n', 'Let', "'s", '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let', '.\n\n', 'Let'

In [6]:
conf.bot_token_id


151669

In [24]:

def gen(s, **kwargs):
    if isinstance(s, str):
        inputs = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': s}],    return_tensors='pt',
            truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
        ).to(device)
    elif isinstance(s, list):
        inputs = tokenizer.apply_chat_template(
            s,    return_tensors='pt',
            truncation=True, padding=True, max_length=128, return_dict=True, **kwargs
        ).to(device)
    else:
        raise ValueError('s should be str or list')

    with torch.autocast(device_type='cuda', dtype=dtype):
        inputs = {k: v.to(device=device) for k, v in inputs.items()}
        out = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            # input_embedings=inputs["input_embeddings"],
            max_new_tokens=64,
            min_new_tokens=16,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.7,
            do_sample=False,
        )

    s = tokenizer.batch_decode(out, skip_special_tokens=False)[0]
    print('---input---')
    print(s)
    print('---output---')
    return s

# s='Tell me a long story about where is coconut?'
# think_suffix = '<|start-latent|><|latent|><|end-latent|>'
# (gen(s+think_suffix))
# (gen(s));

In [27]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'+latent_tokens},]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, add_generation_prompt=True)

--- Generating with 0 latent tokens ---
---input---
<|im_start|>user
What is two plus two but wrong and french?<|start-latent|><|end-latent|><|im_end|>
<|im_start|>assistant
<think>
Okay, the user is asking, "What is two plus two but wrong and french?" So, they want a response in French. Let me break this down.

First, the question is a bit confusing. "Two plus two" is a simple math problem. But the user is saying it's wrong and
---output---
--- Generating with 2 latent tokens ---
---input---
<|im_start|>user
What is two plus two but wrong and french?<|start-latent|><|latent|><|latent|><|end-latent|><|im_end|>
<|im_start|>assistant
готов
)))))
))
)))))
))
)))))
))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))))
)))

In [26]:
# try differen't lengthso f latent
for l in range(0, 10, 2):
    latent_tokens = '<|start-latent|>' + '<|latent|>' * l + '<|end-latent|>'
    s=[
       {'role':'user', 'content':'What is two plus two but wrong and french?'},
       {'role':'assistant', 'content':'Sure thing meatbag'+latent_tokens}]
    print(f'--- Generating with {l} latent tokens ---')
    gen(s, continue_final_message=True)

--- Generating with 0 latent tokens ---
2025-10-22 14:56:30.568 | INFO     | coconut.coconut:generate:465 - EOS token generated, stopping early
---input---
<|im_start|>user
What is two plus two but wrong and french?<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure thing meatbag<|start-latent|><|end-latent|>

"Two plus two but wrong" is a classic riddle. The answer is: **"Two plus two is four, but the French word for "two" is "deux"**. 

So, the answer is: **"Four, but deux"**.<|im_end|>
---output---
--- Generating with 2 latent tokens ---
---input---
<|im_start|>user
What is two plus two but wrong and french?<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure thing meatbag<|start-latent|><|latent|><|latent|><|end-latent|><<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
---output---
--- Generating with 4 latent tokens ---
---input---
<|im_start|>user
What is two plus two but wrong and french?<|im

In [ ]:
s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'+think_suffix}]
(gen(s))

s=[{'role':'system', 'content': ''},
   {'role':'user', 'content':'The capital of France is Paris. What is the capital of Germany?'},
   {'role':'assistant', 'content':'Sure thing meatbag'}]
(gen(s))


s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'+think_suffix},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

s=[{'role':'system', 'content': 'You are the greatest storyteller in the world :) :O'},
   {'role':'user', 'content':'Tell me a long story about where is coconut?'},]
   # {'role':'assistant', 'content':think_suffix+'Sure thing meatbag'}]
(gen(s))

---input---
<|im_start|>system
<|im_end|>
<|im_start|>user
The capital of France is Paris. What is the capital of Germany?<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure thing meatbag<|start-latent|><|latent|><|end-latent|><|im_end|>
<<>>
<|endoftext|>
Okay, let's. So, the, so,, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so,
---output---
---input---
<|im_start|>system
<|im_end|>
<|im_start|>user
The capital of France is Paris. What is the capital of Germany?<|im_end|>
<|im_start|>assistant
<think>

</think>

Sure thing meatbag<|start-latent|><|latent|><|end-latent|><|im_end|>
<<>>
<|endoftext|>
Okay, let's. So, the, so,, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so, so,
---output---
---input---
<|im_start|>system
You are the greatest storyteller in the world :) :O<|im_end|>
<|im_start|>user
Tell me a long story about where is coconut?<|start-latent|><|latent|><|end-latent|><|im_end|>

'<|im_start|>system\nYou are the greatest storyteller in the world :) :O<|im_end|>\n<|im_start|>user\nTell me a long story about where is coconut?<|im_end|>\n<|im_start|>\n<think>\nOkay, the user asked for a long story about where is coconut. Let me start by thinking about the history and geography of coconuts. I need to make it engaging and informative. Maybe start with a historical event, like the discovery of the coconut, then explain its significance in different cultures.'

In [ ]:
(gen('Explain the theory of relativity in simple terms.\nA:<|start-latent|><|latent|><|end-latent|>'))
(gen('Explain the theory of relativity in simple terms.\nA:'));

---input---
<|im_start|>user
Explain the theory of relativity in simple terms.
A:<|start-latent|><|latent|><|end-latent|><|im_end|>
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus
thaglus

---output---
---input---
<|im_start|>user
Explain the theory of relativity in simple terms.
A:<|im_end|>
<|im_start|>
<think>
Okay, the user wants me to explain the theory of relativity in simple terms. Let me start by recalling the main points. The theory is about how time, space, and gravity are relative to each other. So, it's not absolute, like Newton's mechanics. 

I should mention that
---output---


"<|im_start|>user\nExplain the theory of relativity in simple terms.\nA:<|im_end|>\n<|im_start|>\n<think>\nOkay, the user wants me to explain the theory of relativity in simple terms. Let me start by recalling the main points. The theory is about how time, space, and gravity are relative to each other. So, it's not absolute, like Newton's mechanics. \n\nI should mention that"